# Probability calibration

When performing classification you often want not only to predict the class label, but also obtain a probability of the respective label. This probability gives you some kind of confidence on the prediction. Some models can give you poor estimates of the class probabilities and some even do not support probability prediction (e.g., some instances of [`SGDClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html#sklearn.linear_model.SGDClassifier "sklearn.linear_model.SGDClassifier")). The calibration module allows you to better calibrate the probabilities of a given model, or to add support for probability prediction.

Well calibrated classifiers are probabilistic classifiers for which the output of the [predict_proba](https://scikit-learn.org/stable/glossary.html#term-predict_proba) method can be directly interpreted as a confidence level. For instance, a well calibrated (binary) classifier should classify the samples such that among the samples to which it gave a [predict_proba](https://scikit-learn.org/stable/glossary.html#term-predict_proba) value close to, say, 0.8, approximately 80% actually belong to the positive class.

Before we show how to re-calibrate a classifier, we first need a way to detect how good a classifier is calibrated.

## Calibration curves

Calibration curves, also referred to as _reliability diagrams_ (Wilks 1995 [[2]](https://scikit-learn.org/stable/modules/calibration.html#id14)), compare how well the probabilistic predictions of a binary classifier are calibrated. It plots the frequency of the positive label (to be more precise, an estimation of the _conditional event probability_ ) on the y-axis against the predicted probability [predict_proba](https://scikit-learn.org/stable/glossary.html#term-predict_proba) of a model on the x-axis. The tricky part is to get values for the y-axis. In scikit-learn, this is accomplished by binning the predictions such that the x-axis represents the average predicted probability in each bin. The y-axis is then the _fraction of positives_ given the predictions of that bin, i.e. the proportion of samples whose class is the positive class (in each bin).

The top calibration curve plot is created with [`CalibrationDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibrationDisplay.html#sklearn.calibration.CalibrationDisplay.from_estimator "sklearn.calibration.CalibrationDisplay.from_estimator"), which uses [`calibration_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.calibration_curve.html#sklearn.calibration.calibration_curve "sklearn.calibration.calibration_curve") to calculate the per bin average predicted probabilities and fraction of positives. [`CalibrationDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibrationDisplay.html#sklearn.calibration.CalibrationDisplay.from_estimator "sklearn.calibration.CalibrationDisplay.from_estimator") takes as input a fitted classifier, which is used to calculate the predicted probabilities. The classifier thus must have [predict_proba](https://scikit-learn.org/stable/glossary.html#term-predict_proba) method. For the few classifiers that do not have a [predict_proba](https://scikit-learn.org/stable/glossary.html#term-predict_proba) method, it is possible to use [`CalibratedClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html#sklearn.calibration.CalibratedClassifierCV "sklearn.calibration.CalibratedClassifierCV") to calibrate the classifier outputs to probabilities.

The bottom histogram gives some insight into the behavior of each classifier by showing the number of samples in each predicted probability bin.

![Figure: Comparing Calibration Curves](https://scikit-learn.org/stable/_images/sphx_glr_plot_compare_calibration_001.png)

[`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression "sklearn.linear_model.LogisticRegression") is more likely to return well calibrated predictions by itself as it has a canonical link function for its loss, i.e. the logit-link for the [Log loss](https://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). In the unpenalized case, this leads to the so-called **balance property**, see [[8]](https://scikit-learn.org/stable/modules/calibration.html#id20) and [Logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression). In the plot above, data is generated according to a linear mechanism, which is consistent with the [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression "sklearn.linear_model.LogisticRegression") model (the model is ‘well specified’), and the value of the regularization parameter `C` is tuned to be appropriate (neither too strong nor too low). As a consequence, this model returns accurate predictions from its `predict_proba` method. In contrast to that, the other shown models return biased probabilities; with different biases per model.

[`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB "sklearn.naive_bayes.GaussianNB") (Naive Bayes) tends to push probabilities to 0 or 1 (note the counts in the histograms). This is mainly because it makes the assumption that features are conditionally independent given the class, which is not the case in this dataset which contains 2 redundant features.

[`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier "sklearn.ensemble.RandomForestClassifier") shows the opposite behavior: the histograms show peaks at probabilities approximately 0.2 and 0.9, while probabilities close to 0 or 1 are very rare. An explanation for this is given by Niculescu-Mizil and Caruana [[3]](https://scikit-learn.org/stable/modules/calibration.html#id15): “Methods such as bagging and random forests that average predictions from a base set of models can have difficulty making predictions near 0 and 1 because variance in the underlying base models will bias predictions that should be near zero or one away from these values. Because predictions are restricted to the interval [0,1], errors caused by variance tend to be one-sided near zero and one. For example, if a model should predict  for a case, the only way bagging can achieve this is if all bagged trees predict zero. If we add noise to the trees that bagging is averaging over, this noise will cause some trees to predict values larger than 0 for this case, thus moving the average prediction of the bagged ensemble away from 0. We observe this effect most strongly with random forests because the base-level trees trained with random forests have relatively high variance due to feature subsetting.” As a result, the calibration curve shows a characteristic sigmoid shape, indicating that the classifier could trust its “intuition” more and return probabilities closer to 0 or 1 typically.

[`LinearSVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html#sklearn.svm.LinearSVC "sklearn.svm.LinearSVC") (SVC) shows an even more sigmoid curve than the random forest, which is typical for maximum-margin methods (compare Niculescu-Mizil and Caruana [[3]](https://scikit-learn.org/stable/modules/calibration.html#id15)), which focus on difficult to classify samples that are close to the decision boundary (the support vectors).

## Calibrating a classifier

Calibrating a classifier consists of fitting a regressor (called a calibrator) that maps the output of the classifier (as given by decision_function or predict_proba) to a calibrated probability in [0, 1]. Denoting the output of the classifier for a given sample by $f_i$, the calibrator tries to predict the conditional event probability $P(y_i = 1 | f_i)$.

Ideally, the calibrator is fit on a dataset independent of the training data used to fit the classifier in the first place. This is because performance of the classifier on its training data would be better than for novel data. Using the classifier output of training data to fit the calibrator would thus result in a biased calibrator that maps to probabilities closer to 0 and 1 than it should.

The [`CalibratedClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html#sklearn.calibration.CalibratedClassifierCV "sklearn.calibration.CalibratedClassifierCV") class is used to calibrate a classifier.

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Base Model: Addresses structural imbalance by adjusting the loss function.
# This outputs distorted, uncalibrated scores.
base_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000
)

: 

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# 2. Calibrator: Maps the uncalibrated scores back to true probabilities.
# cv=5 means it trains 5 base models and 5 calibrators, creating an ensemble.
# method='isotonic' applies a non-parametric monotonic function to correct the curve.
calibrated_model = CalibratedClassifierCV(
    estimator=base_model,
    method='isotonic',
    cv=5 
)

: 

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_classes=2,
    n_informative=2,
    random_state=1
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y, # Balance the classes in train and test sets
    test_size=0.2,
    random_state=42
)

In [ ]:
# Fit the entire nested pipeline
calibrated_model.fit(X_train, y_train)

print(f"Optimal Threshold on True Probabilities: {calibrated_model.best_threshold_:.4f}")